In [ ]:
#pip install chromadb

In [2]:
import chromadb

In [3]:
chromadb.__version__

'0.5.23'

--------------------------
#### ChromDB

- add TF-IDF vectors into ChromaDB
- Query the database
----------------------------

In [4]:
import pandas as pd

Use **IMDB** dataset

In [6]:
# Load the IMDb dataset
file_path = r'D:\AI-DATASETS\02-MISC-large\IMDB Dataset.csv'
df = pd.read_csv(file_path)

In [7]:
df.shape

(50000, 2)

In [8]:
# Take only the first 1000 reviews
reviews = df['review'].sample(1000).tolist()

#### Generate TF-IDF Vectors
- Use scikit-learn's TfidfVectorizer to generate TF-IDF vectors for the movie reviews.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [10]:
# Initialize the TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=1000)  # Limiting to 1000 features for efficiency

In [11]:
# Generate TF-IDF vectors for the reviews
tfidf_matrix = vectorizer.fit_transform(reviews)

In [12]:
import chromadb

from chromadb.config import Settings

In [13]:
# Initialize ChromaDB client
client = chromadb.Client(Settings(allow_reset = True))

In [14]:
# List all collections
collections = client.list_collections()
print([collection.name for collection in collections])

[]


In [15]:
# Create a collection to store TF-IDF vectors
collection_name = client.get_or_create_collection("imdb_reviews")

In [16]:
collection_name.count()

0

In [17]:
# Attempt to delete the collection
# try:
#     client.delete_collection(name=collection_name)  # Pass the name as a keyword argument
#     print(f"Collection '{collection_name}' deleted successfully.")
# except Exception as e:
#     print(f"Error deleting collection: {e}")

In [18]:
# Convert TF-IDF matrix to dense array and insert into ChromaDB
tfidf_dense = tfidf_matrix.toarray()

In [19]:
# Add each review vector into ChromaDB
# takes abt 1 min
for idx, vector in enumerate(tfidf_dense):
    collection_name.add(
        ids       =[str(idx)],                  # Unique ID for each review
        embeddings=[vector],                    # The TF-IDF vector
        metadatas =[{"review": reviews[idx]}],  # Store the actual review
    )

- ids: Unique identifier for each review.
- embeddings: The TF-IDF vectors.
- metadatas: Metadata like the actual review text, which will be retrieved.

#### Querying ChromaDB with TF-IDF
- query ChromaDB to retrieve similar reviews using a TF-IDF-based retriever.

In [20]:
def query_chromadb(query_text, top_k=5):
    # Convert the query to a TF-IDF vector
    query_vector = vectorizer.transform([query_text]).toarray()[0]
    
    # Perform similarity search in ChromaDB
    results = collection_name.query(
        query_embeddings=[query_vector],  # The query vector
        n_results       =top_k  # Number of results to return
    )
    
    return results

In [21]:
# Example query
query_text = "I love movies about space adventures"
result     = query_chromadb(query_text)

In [22]:
type(result)

dict

In [23]:
result.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'data', 'metadatas', 'distances', 'included'])

In [24]:
result['ids']

[['71', '510', '90', '340', '375']]

In [25]:
result['distances']

[[1.694220781326294,
  1.7157323360443115,
  1.718361735343933,
  1.7490310668945312,
  1.7573257684707642]]

In [26]:
for idx, review in enumerate(result['metadatas'][0]):
    print(review['review'])
    print('---')

If you would like to see a film of different kind, if you feel the Love in your heart, even if you miss the Lord, this film makes you think. Although Georges is mentally handicapped, you can see the ultimate intelligence at the end, when love gives you directions not the brain. I am not emotional, but this film makes you feel the human being. The film is as good as Forrest Gump in my belief. The foreign movies are sometimes more interesting, yet there is not enough advertisement to make them popular. "Rang-e khoda" (The Color of The God) by Majid Majidi is another example of such foreign movies, almost with similar taste.
---
Not good. Mostly because you don't give a damn about what happens to all these people. Some comments : 1. I am tired of seeing governesses who never talk to their pupils, never teach them anything and take a tired and annoyed look whenever the said pupil, who of course has been won over in the space of 4 seconds, says something 2. Fine, so Rosina has a father comp

using **books CSV**

In [23]:
# Load the books csv
# https://www.kaggle.com/datasets/saurabhbagchi/books-dataset/data
file_path = r'D:\AI-DATASETS\02-MISC-large\books.csv'
df = pd.read_csv(file_path, encoding='ISO-8859-1', sep=';', on_bad_lines='skip', low_memory=False)

In [24]:
df.shape

(271360, 8)

In [25]:
df.drop(['Image-URL-S', 'Image-URL-M',	'Image-URL-L'], axis=1, inplace=True)

In [26]:
df.sample(10)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
34972,0373264402,A Romantic Way to Die (Worldwide Library Myste...,Bill Crider,2002,Worldwide Library
34654,0830713662,Curious Waltz of the Working Woman,Karen Scalf Linamen,1990,Gospel Light Pubns
166008,0061054852,Majipoor Chronicles : Majipoor Chronicles (Maj...,Robert Silverberg,1996,Eos
17307,0312203594,Alice's Tulips,Sandra Dallas,2000,St. Martin's Press
184128,037309955X,Man For Mom (That Special Woman! The Family Wa...,Gina Ferris Wilkins,1995,Silhouette
262893,906006125X,D is for Dutch;: An insider's Holland,Jules B Farber,1972,Paris/Manteau
18022,0373501757,Man With A Past (Montana Mavericks #11) (Monta...,Tisha Hamilton,1995,Silhouette
77765,0060194278,"Lighthouse Stevensons, The",Bella Bathurst,1999,HarperCollins Publishers
169134,0807077143,The Jefferson Bible,Thomas Jefferson,2001,Beacon Press
45050,0823073335,"Landscape Graphics: Plan, Section, and Perspec...",Grant W. Reid,2002,Watson-Guptill Publications


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271360 non-null  object
 1   Book-Title           271360 non-null  object
 2   Book-Author          271358 non-null  object
 3   Year-Of-Publication  271360 non-null  object
 4   Publisher            271358 non-null  object
dtypes: object(5)
memory usage: 10.4+ MB


In [28]:
import string,re
#import spacy
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [29]:
def preprocess_text(text):
    #print(f"Original text: {text}")
    
    text = text.lower()  # Lowercasing
    #print(f"Lowercased text: {text}")
    
    # Remove all punctuation except '&'
    text = text.translate(str.maketrans('', '', string.punctuation.replace('&', '')))
    #print(f"Without punctuation (keeping '&'): {text}")
    
    text = text.strip()  # Remove leading/trailing whitespace
    text = re.sub(r'\s+', ' ', text)  # Remove excessive whitespace using regex
    #print(f"Without excessive whitespace: {text}")
    
    # Normalize &amp; if it exists
    text = re.sub(r'&amp;', 'and', text)
    #print(f"After replacing '&amp;': {text}")
    
    # Replace any remaining & with 'and'
    text = text.replace('&', 'and')
    #print(f"After replacing '&': {text}")
    
    return text

In [30]:
# Example usage
sample_text = "This is an example   with   excessive  & whitespace!"
cleaned_text = preprocess_text(sample_text)
print(cleaned_text)

this is an example with excessive and whitespace


In [31]:
# Drop rows with any null values
df_cleaned = df.dropna()

In [32]:
%%time
# Apply the preprocessing
df_cleaned['text'] = df_cleaned['Book-Title'] + ' ' + df_cleaned['Book-Author'] + ' ' + df_cleaned['Publisher']
df_cleaned['text'] = df_cleaned['text'].apply(preprocess_text)

<timed exec>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


CPU times: total: 3.17 s
Wall time: 3.34 s


<timed exec>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [33]:
df_cleaned.shape

(271356, 6)

In [34]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 271356 entries, 0 to 271359
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271356 non-null  object
 1   Book-Title           271356 non-null  object
 2   Book-Author          271356 non-null  object
 3   Year-Of-Publication  271356 non-null  object
 4   Publisher            271356 non-null  object
 5   text                 271356 non-null  object
dtypes: object(6)
memory usage: 14.5+ MB


In [35]:
df_cleaned_samples = df_cleaned.sample(2500)

In [36]:
# Initialize the TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=10000)

In [37]:
# Generate TF-IDF vectors for the reviews
tfidf_matrix = vectorizer.fit_transform(df_cleaned_samples.text)

In [38]:
# Reset the client, which clears all collections and data
client.reset()

print("ChromaDB client has been reset.")

ChromaDB client has been reset.


In [39]:
# List all collections
collections = client.list_collections()
print([collection.name for collection in collections])

[]


In [40]:
# Create a collection to store TF-IDF vectors
collection_name = client.get_or_create_collection("book_info")

In [41]:
# Delete the collection
# client.delete_collection(collection_name)

# print(f"Collection '{collection_name}' has been deleted.")

In [42]:
# Convert TF-IDF matrix to dense array and insert into ChromaDB
tfidf_dense = tfidf_matrix.toarray()

In [43]:
tfidf_dense.shape

(2500, 8507)

In [45]:
df_cleaned_samples.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'text'],
      dtype='object')

In [44]:
%%time

# takes abt 1-3 mins
# Store the original indices before any resetting
original_indices = df_cleaned_samples.index.tolist()  # Store original indices

# Add each review vector into ChromaDB
for idx, vector in enumerate(tfidf_dense):

    # Use original_indices to fetch metadata
    original_idx = original_indices[idx]
    
    # Construct metadata with book information
    metadata = {
        "Title": df_cleaned_samples.loc[original_idx, 'Book-Title'],          # Book Title
        "Author": df_cleaned_samples.loc[original_idx, 'Book-Author'],        # Book Author
        "Year": df_cleaned_samples.loc[original_idx, 'Year-Of-Publication'],  # Year of Publication
        "Publisher": df_cleaned_samples.loc[original_idx, 'Publisher'],       # Publisher
    }
    
    # Add the vector and metadata to ChromaDB collection
    collection_name.add(
        ids=[str(idx)],                  # Unique ID for each book/review
        embeddings=[vector],             # The TF-IDF vector
        metadatas=[metadata]             # Store book details (metadata)
    )

CPU times: total: 1min 21s
Wall time: 1min


#### Querying ChromaDB with TF-IDF
- query ChromaDB to retrieve similar reviews using a TF-IDF-based retriever.

In [45]:
def query_chromadb(query_text, top_k=3):
    # Convert the query to a TF-IDF vector
    query_vector = vectorizer.transform([query_text]).toarray()[0]
    
    # Perform similarity search in ChromaDB
    results = collection_name.query(
        query_embeddings=[query_vector],  # The query vector
        n_results       =top_k            # Number of results to return
    )
    
    return results

In [46]:
df_cleaned_samples.sample(10)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,text
85212,1570362882,The Unicorn Sonata,Peter S. Beagle,1996,Turner Pub,the unicorn sonata peter s beagle turner pub
57001,0316128228,Handbook for the Soul,Richard Carlson,1996,Back Bay Books,handbook for the soul richard carlson back bay...
240011,051606004X,The Sun's Family of Planets (Rookie Read-About...,Allan Fowler,1992,Children's Press (CT),the suns family of planets rookie readabout sc...
208884,3596127610,Neon - NÃ?Â¤chte.,Sabine Deitmer,1995,"Fischer (Tb.), Frankfurt",neon nãâ¤chte sabine deitmer fischer tb frankfurt
200301,0632053194,Infectious Disease,Barbara Bannister,2000,Blackwell Publishers,infectious disease barbara bannister blackwell...
250193,0030626285,Concise Guide for Writers,Louis E. Glorfeld,1984,Thomson Learning,concise guide for writers louis e glorfeld tho...
255050,0451521765,One Day in the Life of Ivan Denisovich,Aleksandr Isaevich Solzhenitsyn,1998,Signet Book,one day in the life of ivan denisovich aleksan...
213437,0064634213,"The Creative Writer's Handbook: What to Write,...",Isabelle Gibson Ziegler,1975,Harpercollins,the creative writers handbook what to write ho...
206902,0762100834,Watercolor Workbook: A Complete Course in Ten ...,Anne Elsworth,1998,Readers Digest,watercolor workbook a complete course in ten l...
154228,0786926597,Urban Arcana Campaign Setting (d20 Modern),Bill Slavicsek,2003,Wizards of the Coast,urban arcana campaign setting d20 modern bill ...


In [47]:
# Example query
query_text = "Embers of Dawn"
result = query_chromadb(query_text)

In [48]:
type(result)

dict

In [49]:
result.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [50]:
result['ids']

[['112', '509', '380']]

In [51]:
result['distances']

[[1.2720093727111816, 1.3067282438278198, 1.3168532848358154]]

In [52]:
for idx, info in enumerate(result['metadatas'][0]):
    print(info['Author'])
    
    print(info['Title'])
    print(info['Publisher'])
    print(info['Year'])
    print('---')

N. Scott Momaday
House Made of Dawn (Perennial Classics)
Perennial Classics
1999
---
Shannon Drake
Seize the Dawn (Zebra Historical Romance)
Kensington Publishing Corporation
2001
---
Meagan McKinney
Till Dawn Tames the Night
Dell Publishing Company
1994
---


#### Why we used ChromaDB (vector database)

`Efficient Storage of High-Dimensional Data`
- ChromaDB is optimized for storing and retrieving embeddings (like those produced by TF-IDF, word2vec, or other vectorization methods) that can be high-dimensional and sparse. This optimization helps in efficiently managing large datasets.

`Fast Similarity Search`
- It provides efficient querying capabilities to perform similarity searches. For instance, you can quickly find similar documents or items based on their vector representations, which is crucial in recommendation systems and search applications.

`Metadata Storage`
- Alongside the embeddings, ChromaDB allows for the storage of associated metadata, making it easier to retrieve relevant information about the embeddings (like book titles, authors, publication years, etc.) in addition to the embeddings themselves.

`Support for Various Embedding Types`
- ChromaDB can handle different types of embeddings, whether they are generated from TF-IDF, neural networks, or other methods, allowing flexibility in how you represent your data.

`Example Use Cases`
- **Recommendation Systems**: Finding similar books based on user preferences.
- **Search Engines**: Enabling fast searches for documents or products based on vector similarity.
- **Natural Language Processing Applications**: Enhancing tasks like semantic search, document clustering, and classification.


